# PART D

In [ ]:
%pip install -q pandas numpy scikit-learn nltk transformers

In [ ]:
import pandas as pd
import gzip
import json
import numpy as np

In [ ]:
documents = []

with gzip.open('../data/c4-train.00000-of-01024-30K.json.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            documents.append(json.loads(line))

In [ ]:
print(documents[0])  # Print the first document to verify the content

In [ ]:
print(len(documents))  # Print the number of documents loaded

In [ ]:
df = pd.DataFrame(documents)
df

In [ ]:
df1 = df['text']  # Extract the 'text' column as a list of strings
df1

In [ ]:
print("Number of documents:", len(df1))

In [ ]:
df1_lowercase = df1.str.lower()  # Convert all text to lowercase
df1_lowercase

In [ ]:
import re

def tokenize_normalized(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    tokens = re.findall(r"\b\w+\b", text)
    return tokens

In [ ]:
df1_tokens = df1.apply(tokenize_normalized) 
df1_tokens

In [ ]:
df1_tokens[1321]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(
    tokenizer=str.split,
    token_pattern=None,
    lowercase=False
)

X_count = count_vectorizer.fit_transform(
    df1_tokens.apply(lambda tokens: ' '.join(tokens))
)

In [ ]:
print(X_count)

In [ ]:
print(X_count.shape)
print(X_count[:3].toarray().max()) # Print the maximum count value

In [ ]:
print("Matrix shape:", X_count.shape)
print("Vocabulary size:", len(count_vectorizer.vocabulary_))

In [ ]:
vocabulary = count_vectorizer.get_feature_names_out()
vocabulary

In [ ]:
#Tính độ thưa
nnz = X_count.nnz  # Số zero entries trong ma trận sparse
print(1 - nnz / (X_count.shape[0] * X_count.shape[1]))

In [ ]:
import numpy as np

# Df
document_frequency = np.asarray((X_count > 0).sum(axis=0)).ravel()
top_df_indices = np.argsort(document_frequency)[::-1][:20]

top_df = pd.DataFrame({
    'term': vocabulary[top_df_indices],
    'document_frequency': document_frequency[top_df_indices]
})
top_df

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer()
X_tfidf = tfidf_transformer.fit_transform(X_count)
idf = tfidf_transformer.idf_
top_idf_indices = np.argsort(idf)[::-1][:20]

top_idf = pd.DataFrame({
    'term': vocabulary[top_idf_indices],
    'idf': idf[top_idf_indices]
})
top_idf

In [ ]:
selected_doc_index = 1321
selected_tfidf = X_tfidf[selected_doc_index].toarray().ravel()
top_tfidf_indices = np.argsort(selected_tfidf)[::-1][:20]

top_tfidf = pd.DataFrame({
    'term': vocabulary[top_tfidf_indices],
    'tfidf': selected_tfidf[top_tfidf_indices]
})
top_tfidf

In [ ]:
df1[1321]  # Print the original text of the selected document to verify the content

- Nhìn vào document 1321, có thể thấy "the" là một từ có df cao nhất trong corpus, nhưng vẫn có chỉ số tfidf xếp thứ 4 trong document 1321 vì có tf đủ lớn. Như vậy, một term có df cao vẫn có thể có tfidf cao.
- Một term có idf cao không nhất thiết có tf-idf cao trong mọi document vì nó còn phụ vào tf của term đó trong document đó vì công thức tfidf phụ thuộc vào cả tf lẫn idf. Trường hợp tf = 0 (không xuất hiện trong doc) thì dù có idf cao nhưng vẫn có tfidf bằng 0.

# PART F

In [ ]:
# Giải phóng các ma trận và danh sách token của Part D để dành RAM cho các pipeline
del df1_tokens, X_count, X_tfidf, count_vectorizer, tfidf_transformer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# Pipeline A: lowercase + tokenize, giữ punctuation
def tokenize_pipeline_A(text):
    return word_tokenize(text.lower(), preserve_line=True)

# Pipeline B: lowercase + chuẩn hóa punctuation + bỏ stopwords
def tokenize_pipeline_B(text):
    text = re.sub(r"[^\w\s]", " ", text.lower())
    tokens = re.findall(r"\b[a-zA-Z]+\b", text)
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]

# Pipeline C: tokenizer WordPiece đã huấn luyện sẵn của BERT
tokenizer_C = AutoTokenizer.from_pretrained('bert-base-uncased')
def tokenize_pipeline_C(text):
    return tokenizer_C.tokenize(text)

print('A:', tokenize_pipeline_A(df1.iloc[0])[:20])
print('B:', tokenize_pipeline_B(df1.iloc[0])[:20])
print('C:', tokenize_pipeline_C(df1.iloc[0])[:20])

In [ ]:
# Chia cùng corpus thành train/test để đo OOV theo cùng điều kiện
train_docs, test_docs = train_test_split(df1, test_size=0.2, random_state=42)

def evaluate_pipeline(name, tokenizer):
    vectorizer = CountVectorizer(tokenizer=tokenizer, token_pattern=None, lowercase=False)
    X_train = vectorizer.fit_transform(train_docs)
    vocabulary = vectorizer.vocabulary_
    token_count = 0
    oov_count = 0
    for document in test_docs:
        document_tokens = tokenizer(document)
        token_count += len(document_tokens)
        oov_count += sum(
            token not in vocabulary or (tokenizer is tokenize_pipeline_C and token == tokenizer_C.unk_token)
            for token in document_tokens
        )
    n_documents, n_terms = X_train.shape
    return {
        'Pipeline': name,
        'Vocabulary size': n_terms,
        'Average tokens/document': X_train.sum() / n_documents,
        'Matrix sparsity': 1 - X_train.nnz / (n_documents * n_terms),
        'OOV rate': oov_count / token_count if token_count else 0
    }

metrics = pd.DataFrame([
    evaluate_pipeline('Pipeline A', tokenize_pipeline_A),
    evaluate_pipeline('Pipeline B', tokenize_pipeline_B),
    evaluate_pipeline('Pipeline C', tokenize_pipeline_C)
])
metrics

# PART G — Tìm kiếm tài liệu

Dùng các query gợi ý trong đề; mỗi pipeline có vocabulary và TF-IDF index riêng. ID tài liệu là chỉ số dòng corpus bắt đầu từ 0.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

evaluation_set = pd.read_csv('evaluation_set.csv').fillna('')
queries = evaluation_set['query'].tolist()
if not 5 <= len(queries) <= 10:
    raise ValueError('evaluation_set.csv cần có từ 5 đến 10 query.')
pipelines = {'Pipeline A': tokenize_pipeline_A, 'Pipeline B': tokenize_pipeline_B, 'Pipeline C': tokenize_pipeline_C}

result_rows = []
for pipeline_name, tokenizer in pipelines.items():
    vectorizer = CountVectorizer(tokenizer=tokenizer, token_pattern=None, lowercase=False)
    X_count_search = vectorizer.fit_transform(df1)
    tfidf_transformer = TfidfTransformer()
    X_docs_tfidf = tfidf_transformer.fit_transform(X_count_search)
    X_queries_tfidf = tfidf_transformer.transform(vectorizer.transform(queries))
    scores = cosine_similarity(X_queries_tfidf, X_docs_tfidf)
    for query_index, query in enumerate(queries):
        top_indices = scores[query_index].argsort()[-5:][::-1]
        for rank, document_id in enumerate(top_indices, start=1):
            result_rows.append({
                'pipeline': pipeline_name, 'query': query, 'rank': rank,
                'document_id': int(document_id),
                'similarity': float(scores[query_index, document_id]),
                'document_preview': str(df1.iloc[document_id])[:250].replace('\n', ' '),
                'relevant': ''
            })
    del X_count_search, X_docs_tfidf, X_queries_tfidf, scores, vectorizer, tfidf_transformer

results = pd.DataFrame(result_rows)
results.to_csv('results.csv', index=False)
results.head(15)

# PART H — Đánh giá

Tự thẩm định relevance của kết quả rồi điền các ID liên quan vào `evaluation_set.csv`, phân tách bằng dấu chấm phẩy. Không tạo nhãn chỉ dựa trên thứ hạng của hệ thống.

In [ ]:
evaluation_set = pd.read_csv('evaluation_set.csv').fillna('')
if evaluation_set['relevant_document_ids'].str.strip().eq('').any():
    raise ValueError('Hãy tự gán nhãn relevant_document_ids trong evaluation_set.csv trước khi tính metric.')

metric_rows = []
for pipeline_name in results['pipeline'].unique():
    reciprocal_ranks, precisions, recalls = [], [], []
    for _, qrel in evaluation_set.iterrows():
        relevant_ids = {int(value) for value in str(qrel['relevant_document_ids']).split(';') if value.strip()}
        if not relevant_ids or any(doc_id < 0 or doc_id >= len(df1) for doc_id in relevant_ids):
            raise ValueError(f"ID tài liệu relevance không hợp lệ cho query: {qrel['query']}")
        retrieved = results[(results['pipeline'] == pipeline_name) & (results['query'] == qrel['query'])].sort_values('rank')
        hits = retrieved['document_id'].isin(relevant_ids).tolist()
        precisions.append(sum(hits) / 5)
        recalls.append(sum(hits) / len(relevant_ids))
        first_hit_rank = next((rank for rank, is_relevant in enumerate(hits, start=1) if is_relevant), None)
        reciprocal_ranks.append(1 / first_hit_rank if first_hit_rank else 0)
    metric_rows.append({'pipeline': pipeline_name, 'Precision@5': sum(precisions) / len(precisions), 'Recall@5': sum(recalls) / len(recalls), 'MRR': sum(reciprocal_ranks) / len(reciprocal_ranks)})

evaluation_metrics = pd.DataFrame(metric_rows)
metrics = metrics.merge(evaluation_metrics, left_on='Pipeline', right_on='pipeline').drop(columns='pipeline')
evaluation_metrics.to_csv('evaluation_metrics.csv', index=False)
metrics

## PART I — Phân tích lỗi

Sau khi kiểm tra nhãn và kết quả, phân tích 2 query tốt và 2 query kém. Với mỗi query, ghi tài liệu liên quan mong đợi, tài liệu đã lấy về, term đóng góp vào similarity, lexical overlap và nguyên nhân thành công/thất bại. Chọn một failure case để giải thích kỹ.

## PART J — Từ failure đến representation tiếp theo

Dựa trên failure case thực tế, nêu giả thuyết về representation có thể nhận ra quan hệ ngữ nghĩa dù hai cách diễn đạt không dùng chung từ. Ghi reflection trong `reflection.md`.

# Xuất notebook kèm toàn bộ output

Chạy cell kế tiếp sau khi đã chạy các thí nghiệm. Colab sẽ tải xuống một bản `.ipynb` có lưu output hiện tại của các cell.

In [ ]:
import json
from google.colab import _message, files

def export_notebook_outputs(filename='lab01_experiments_with_outputs.ipynb'):
    notebook = _message.blocking_request(
        'get_ipynb', request='', timeout_sec=30
    )
    if notebook is None:
        raise RuntimeError('Colab không trả về nội dung notebook.')

    if isinstance(notebook, str):
        notebook = json.loads(notebook)
    if not isinstance(notebook, dict) or 'cells' not in notebook:
        raise ValueError('Dữ liệu nhận được không phải notebook hợp lệ.')

    with open(filename, 'w', encoding='utf-8') as output_file:
        json.dump(notebook, output_file, ensure_ascii=False, indent=1)

    files.download(filename)
    return filename

export_notebook_outputs()